In [ ]:
# ── CONFIGURATION ───────────────────────────────────────────
# Replace the placeholders below with your actual Azure credentials.
# Never commit real credentials to version control.
# Use environment variables or Azure Key Vault in production.


# ── CONNECT TO ADLS Gen2 GOLD LAYER ─────────────────────────
storage_account = "YOUR_STORAGE_ACCOUNT_NAME"
storage_key     = "Cannot share the storage key here for security reasons. Please replace with your own storage key."
container       = "gold-layer"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

gold_base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
print("Connected to ADLS Gen2 Gold Layer")

In [ ]:
# ── RECONNECT USING WASBS PROTOCOL + SAS TOKEN ──────────────
storage_account = "YOUR_STORAGE_ACCOUNT_NAME"
sas_token       = "cannot share the SAS token here for security reasons. Please replace with your own SAS token."
container       = "gold-layer"


spark.conf.set(
    f"fs.azure.sas.{container}.{storage_account}.blob.core.windows.net",
    sas_token
)

gold_base = f"wasbs://{container}@{storage_account}.blob.core.windows.net"
print("Reconnected via WASBS + SAS Token")

# ── FIX TIMESTAMP PRECISION BEFORE READING ──────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

def fix_timestamps(df):
    for col_name, dtype in df.dtypes:
        if 'timestamp' in col_name.lower():
            df = df.withColumn(col_name, F.col(col_name).cast(TimestampType()))
    return df

# ── READ ALL 3 GOLD TABLES ───────────────────────────────────
df_patients = fix_timestamps(spark.read.option("mergeSchema", "false").parquet(f"{gold_base}/patient_risk_summary/patient_risk_summary.parquet"))
df_hourly   = fix_timestamps(spark.read.option("mergeSchema", "false").parquet(f"{gold_base}/hourly_aggregations/hourly_aggregations.parquet"))
df_summary  = fix_timestamps(spark.read.option("mergeSchema", "false").parquet(f"{gold_base}/icu_summary/icu_summary.parquet"))

print(f"Patient Risk Summary : {df_patients.count()} rows")
print(f"Hourly Aggregations  : {df_hourly.count()} rows")
print(f"ICU Summary          : {df_summary.count()} rows")

In [ ]:
# ── WRITE GOLD TABLES TO LAKEHOUSE AS DELTA TABLES ──────────
print("Writing to Lakehouse Delta tables...")

# Table 1: Patient Risk Summary
df_patients.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("patient_risk_summary")
print("patient_risk_summary table created")

# Table 2: Hourly Aggregations
df_hourly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("hourly_aggregations")
print("hourly_aggregations table created")

# Table 3: ICU Summary
df_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("icu_summary")
print("icu_summary table created")

print("\n All Gold tables loaded into Lakehouse!")
print("   Power BI can now connect directly to these tables.")

In [ ]:
# ── VERIFY TABLES EXIST IN LAKEHOUSE ────────────────────────
tables = spark.sql("SHOW TABLES")
tables.show()